In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

In [55]:
df = pd.read_csv("master_features_windowed.csv")

# Define Inputs (X) and Output (y)
X = df[['BMI', 'Red_Variability', 'IR_Variability', 'Red_Skew', 'IR_Skew', 'Red_Kurtosis', 'IR_Kurtosis']]
y = df['Glucose']

# 2. SPLIT THE DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. SCALE THE DATA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [56]:
num_predictors = X_train.shape[1]
fine_gamma = 8.0 / num_predictors
medium_gamma = 0.5 / num_predictors

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    
    "Support Vector Machine (SVR)": SVR(kernel='rbf', C=100, epsilon=0.1),
    "SVR-Fine Gaussian": SVR(kernel='rbf', gamma=fine_gamma, C=10.0, epsilon=0.1),
    "Non-Linear Medium Gaussian SVR": SVR(kernel='rbf', gamma=medium_gamma, C=10.0, epsilon=0.1),
    
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, random_state=42),
    
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
    "CatBoost Regressor": CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05, verbose=0, random_seed=42)
}

In [57]:
best_rmse = float('inf')
best_model = None
best_model_name = None

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    print(f"\n{name}:")
    print(f"  ➡ MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model_name = name


Linear Regression:
  ➡ MAE: 35.12 | RMSE: 41.73 | R2: -0.34

Ridge Regression:
  ➡ MAE: 30.11 | RMSE: 35.48 | R2: 0.03

Lasso Regression:
  ➡ MAE: 28.99 | RMSE: 35.47 | R2: 0.03

ElasticNet:
  ➡ MAE: 35.40 | RMSE: 45.20 | R2: -0.57

K-Nearest Neighbors:
  ➡ MAE: 27.83 | RMSE: 36.79 | R2: -0.04

Support Vector Machine (SVR):
  ➡ MAE: 31.85 | RMSE: 38.09 | R2: -0.12

SVR-Fine Gaussian:
  ➡ MAE: 28.47 | RMSE: 38.10 | R2: -0.12

Non-Linear Medium Gaussian SVR:
  ➡ MAE: 25.27 | RMSE: 35.49 | R2: 0.03

Random Forest:
  ➡ MAE: 24.61 | RMSE: 30.67 | R2: 0.28

Gradient Boosting:
  ➡ MAE: 14.49 | RMSE: 20.90 | R2: 0.66

AdaBoost:
  ➡ MAE: 23.48 | RMSE: 29.86 | R2: 0.31

XGBoost:
  ➡ MAE: 22.77 | RMSE: 27.59 | R2: 0.41

LightGBM:
  ➡ MAE: 27.93 | RMSE: 32.78 | R2: 0.17


c:\Users\Revios\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



CatBoost Regressor:
  ➡ MAE: 21.63 | RMSE: 27.00 | R2: 0.44


In [58]:
joblib.dump(best_model, 'Gradient_Boosting.joblib')
joblib.dump(scaler, 'scaler_GB.joblib')

print(f"\nBest Model: {best_model_name} (RMSE: {best_rmse:.2f})")
print("Saved best model to 'Gradient_Boosting.joblib'")


Best Model: Gradient Boosting (RMSE: 20.90)
Saved best model to 'Gradient_Boosting.joblib'
